# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hamidism/Machine-Learning-intern/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [14]:
from huggingface_hub import login, hf_hub_download
from google.colab import userdata
import pandas as pd

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

content_path = hf_hub_download(repo_id="FlyRank/internship-warehouse", filename="dim_content.parquet", repo_type="dataset")
perf_path = hf_hub_download(repo_id="FlyRank/internship-warehouse", filename="fact_content_daily_performance/month=2026-03/data_0.parquet", repo_type="dataset")

dim_content = pd.read_parquet(content_path)
fact_perf = pd.read_parquet(perf_path)
fact_perf["report_date"] = pd.to_datetime(fact_perf["report_date"])

available = fact_perf[(fact_perf["gsc_data_available"] == True) & (fact_perf["ga4_data_available"] == True)]

page_agg = available.groupby("content_hash_id").agg(
    total_impressions=("gsc_impressions", "sum"),
    total_clicks=("gsc_clicks", "sum"),
    avg_position=("gsc_avg_position", "mean")
).reset_index()
page_agg["ctr"] = page_agg["total_clicks"] / page_agg["total_impressions"].replace(0, pd.NA)

df = page_agg.merge(dim_content[["content_hash_id","word_count","char_count","backlinks",
                                   "search_volume","content_updated_date"]], on="content_hash_id", how="left")
df["days_since_update"] = (pd.Timestamp("2026-03-31") - pd.to_datetime(df["content_updated_date"])).dt.days
df = df.dropna(subset=["avg_position","ctr","days_since_update","search_volume"])
print("Rows ready:", len(df))

Rows ready: 61125


## 1. My rule and its reason codes

Signal 1 verdict: FALSE / INCONCLUSIVE — staleness cannot be reliably tested on this slice.
Only 12,118 of 61,125 pages (19.8%) have a content_updated_date that was actually knowable
as of March 31, 2026 — the rest have update dates in April-July 2026, which is a data quality
issue (the field appears to record a later pipeline event, not the true last-edit date). Within
the usable sample, there's almost no long-tail staleness (only 2 pages beyond 365 days) and CTR
does not show the expected decline with staleness — if anything it moves the opposite direction,
though this is likely noise given the tiny bucket sizes. I'm not using staleness as a primary
input to my rule because I can't trust it for 80% of pages; this is reported as an honest
negative finding rather than worked around.

In [15]:
df["staleness_bucket"] = pd.cut(df["days_since_update"], bins=[-1,90,180,365,10000],
                                  labels=["0-90d","91-180d","181-365d","365d+"])
stale_table = df.groupby("staleness_bucket", observed=True).agg(
    n=("content_hash_id","count"),
    avg_ctr=("ctr","mean"),
    avg_impressions=("total_impressions","mean")
)
print(stale_table)

                      n   avg_ctr  avg_impressions
staleness_bucket                                  
0-90d             11957  0.013610       784.474868
91-180d             159  0.015791      1645.245283
181-365d              2  0.500000         1.000000


In [16]:
df["position_bucket"] = pd.cut(df["avg_position"], bins=[0,3,10,20,50,1000],
                                 labels=["1-3","4-10","11-20","21-50","50+"])
pos_table = df.groupby("position_bucket", observed=True).agg(
    n=("content_hash_id","count"),
    avg_ctr=("ctr","mean")
)
print(pos_table)

                     n   avg_ctr
position_bucket                 
1-3               7622  0.035105
4-10             26425  0.022504
11-20            11901  0.016683
21-50            12622  0.012285
50+               1561  0.017789


In [17]:
print(df["days_since_update"].describe())
print("\nRows with days_since_update > 10000:", (df["days_since_update"] > 10000).sum())
print("Sample of extreme values:")
print(df[df["days_since_update"] > 10000][["content_hash_id","days_since_update"]].head(5))

count    61125.000000
mean       -45.681112
std         43.013348
min        -97.000000
25%        -78.000000
50%        -50.000000
75%        -48.000000
max        303.000000
Name: days_since_update, dtype: float64

Rows with days_since_update > 10000: 0
Sample of extreme values:
Empty DataFrame
Columns: [content_hash_id, days_since_update]
Index: []


In [18]:
print(dim_content["content_updated_date"].describe())
print("\nSample dates:")
print(dim_content[["content_hash_id","content_updated_date"]].head(10))
print("\nMax date:", dim_content["content_updated_date"].max())
print("Min date:", dim_content["content_updated_date"].min())

count         519606
unique           242
top       2026-05-20
freq          204409
Name: content_updated_date, dtype: object

Sample dates:
            content_hash_id content_updated_date
0  content_004de9653278b5a4           2026-07-01
1  content_00dc5efae381b2ab           2026-07-01
2  content_01410f2556c327ac           2026-07-01
3  content_019f27f634053ca7           2026-06-15
4  content_01efa71faea45dcc           2026-06-01
5  content_01fc9e2e57898b55           2026-07-01
6  content_0212158fa61c5fcb           2026-07-01
7  content_023d807c7d922db1           2026-06-13
8  content_026f7405cc253242           2026-07-01
9  content_02c8b23ea5bdb275           2026-07-01

Max date: 2026-07-06
Min date: 2024-10-28


In [19]:
# Drop the old (bad) days_since_update from df before merging with the corrected version
df_clean = df.drop(columns=["days_since_update"])

df_staleness = df_clean.merge(valid_updates[["content_hash_id","days_since_update"]], on="content_hash_id", how="inner")

print(f"Pages available for staleness signal check: {len(df_staleness)}")

df_staleness["staleness_bucket"] = pd.cut(df_staleness["days_since_update"], bins=[-1,90,180,365,10000],
                                            labels=["0-90d","91-180d","181-365d","365d+"])
stale_table = df_staleness.groupby("staleness_bucket", observed=True).agg(
    n=("content_hash_id","count"),
    avg_ctr=("ctr","mean"),
    avg_impressions=("total_impressions","mean")
)
print(stale_table)

Pages available for staleness signal check: 12118
                      n   avg_ctr  avg_impressions
staleness_bucket                                  
0-90d             11957  0.013610       784.474868
91-180d             159  0.015791      1645.245283
181-365d              2  0.500000         1.000000


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [20]:
# Expected CTR per position bucket (from the bucket table above)
expected_ctr = df.groupby("position_bucket", observed=True)["ctr"].transform("mean")
df["ctr_gap"] = expected_ctr - df["ctr"]  # positive = underperforming

def reason_code(row):
    stale = row["days_since_update"] >= 181
    weak_ctr = row["ctr_gap"] > 0
    if stale and weak_ctr: return "STALE_UNDERPERFORM"
    if stale: return "STALE_ONLY"
    if weak_ctr: return "CTR_ONLY"
    return "HEALTHY"

df["reason_code"] = df.apply(reason_code, axis=1)

def action_label(rc):
    return {"STALE_UNDERPERFORM":"refresh", "STALE_ONLY":"monitor",
            "CTR_ONLY":"refresh", "HEALTHY":"protect"}[rc]

df["action"] = df["reason_code"].apply(action_label)

# Score: weight ctr_gap and staleness by search_volume (opportunity size)
df["score"] = (df["ctr_gap"].clip(lower=0) * 0.6 +
               (df["days_since_update"]/365).clip(upper=2) * 0.4) * (df["search_volume"].fillna(0) + 1).apply(lambda x: min(x,1000)/1000 + 0.5)

ranked = df.sort_values("score", ascending=False).reset_index(drop=True)
df["score"] = df["score"] * (df["total_impressions"] >= 50).astype(int)
import os
os.makedirs("work/outputs", exist_ok=True)
ranked[["content_hash_id","score","reason_code","action","avg_position","ctr","days_since_update","search_volume"]].to_csv(
    "work/outputs/baseline_action_score.csv", index=False)

print("Saved", len(ranked), "ranked rows")
ranked.head(20)[["content_hash_id","score","reason_code","action","avg_position","ctr","days_since_update"]]

Saved 61125 ranked rows


,content_hash_id,score,reason_code,action,avg_position,ctr,days_since_update
0,content_4bc0672ea82ded6c,0.275714,CTR_ONLY,refresh,27.750000,0.000000,161
1,content_20483e735917ac4b,0.166359,STALE_ONLY,monitor,1.000000,1.000000,303
2,content_8af31b9bc99a03fd,0.137054,STALE_UNDERPERFORM,refresh,89.000000,0.000000,235
3,content_d6778d346674532b,0.109813,CTR_ONLY,refresh,1.000000,0.000000,166
4,content_41ea70f2b50e9f44,0.103833,CTR_ONLY,refresh,18.333333,0.000000,166
5,content_b960150c6fcecf80,0.103723,CTR_ONLY,refresh,1.454545,0.000000,166
6,content_19daa2f24df1882d,0.103675,CTR_ONLY,refresh,12.491368,0.000000,176
7,content_3dbcd80e408ef2c9,0.101144,CTR_ONLY,refresh,1.400000,0.000000,165
8,content_638e5bf375392714,0.100859,CTR_ONLY,refresh,5.000000,0.000000,161
9,content_bbee067b07e393f6,0.099860,CTR_ONLY,refresh,9.000000,0.000000,166


## 3. Top-20 review
1. content_4bc0672ea82ded6c — action: refresh, reason: CTR_ONLY. Position 27.75 with 0% CTR —
   large gap vs the ~1.2-2.2% expected at that tier. Would be wrong if: total_impressions is
   very low here, making the 0% CTR a small-sample fluke rather than a real problem.

2. content_20483e735917ac4b — action: monitor, reason: STALE_ONLY. Position 1.0 with CTR 1.0
   (100%) — this looks like an extremely low-impression page (maybe just 1 impression, 1 click),
   not a genuinely healthy top result. Would be wrong if: this is being read as "great CTR"
   when it's really a sample-size artifact.

3. content_8af31b9bc99a03fd — action: refresh, reason: STALE_UNDERPERFORM. Position 89 (very
   weak) with 0% CTR and 235 days stale — most defensible pick in the list, multiple signals
   agree. Would be wrong if: position 89 pages realistically never get clicks regardless of
   content quality, making "refresh" the wrong action vs "deprioritize."

4-12. content_d6778d346674532b, content_41ea70f2b50e9f44, content_b960150c6fcecf80,
   content_19daa2f24df1882d, content_3dbcd80e408ef2c9,

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

Weak picks: 16 of the top-20 refresh picks have extremely low total_impressions (1-17),
making their 0% CTR statistically meaningless rather than a real signal — with so few
impressions, 0 clicks is the expected outcome by chance, not evidence of an underperforming
page. Only 4 of the top 20 have enough volume (500+ impressions) to trust the CTR gap as real:
content_19daa2f24df1882d (513), content_50bc2f19f2a7f355 (3,009), content_56ed8db5afa76626
(2,299), and content_a5398cc741369689 (1,003). This is a concrete flaw in the current rule —
it needs a minimum impression floor (e.g., 100+) before trusting a CTR gap, otherwise it
surfaces noise as opportunity.

I also confirmed the days_since_update == 34 pattern is not a small coincidence: 11,359 pages
(18.6% of the full dataset) share this exact value. This is far too large and too precise to
be organic — it strongly suggests content_updated_date reflects a bulk system/import event
for a large share of pages, not genuine per-page editing history. Combined with the earlier
finding that 80% of update dates fall in the future relative to March 2026, this confirms
content_updated_date is not reliable as a staleness signal for this dataset, and I'm
deliberately excluding it from the score for that reason.

Leakage check: Confirmed — the score uses only ctr_gap (from March position/CTR data) and
search_vol

In [22]:
# ranked already has total_impressions — no merge needed
top20 = ranked.head(20)
print(top20[["content_hash_id","ctr","total_impressions"]])

# Confirm the days_since_update==34 cluster size
print("\nPages with days_since_update exactly 34:", (df["days_since_update"]==34).sum())

             content_hash_id       ctr  total_impressions
0   content_4bc0672ea82ded6c  0.000000                  3
1   content_20483e735917ac4b  1.000000                  1
2   content_8af31b9bc99a03fd  0.000000                  1
3   content_d6778d346674532b  0.000000                  1
4   content_41ea70f2b50e9f44  0.000000                  3
5   content_b960150c6fcecf80  0.000000                 11
6   content_19daa2f24df1882d  0.000000                513
7   content_3dbcd80e408ef2c9  0.000000                 11
8   content_638e5bf375392714  0.000000                  1
9   content_bbee067b07e393f6  0.000000                  3
10  content_98c28dea60c39cde  0.000000                  2
11  content_88b2137344e3ee8b  0.000000                  5
12  content_cb65cb0398607ec6  0.000000                 17
13  content_05ae683c527dc32d  0.000000                  3
14  content_55b1b4d1462fe7d3  0.000000                  1
15  content_8a7f512b356a2398  0.000000                 56
16  content_50

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.